# Slooze Data Engineer Challenge — EDA Report
**Author:** Panga Shravan Yadav  
**Dataset:** IndiaMART B2B Product Listings (Scraped)  
**Objective:** Exploratory Data Analysis on scraped B2B marketplace data to uncover pricing patterns, supplier distribution, and category insights.

code
#VSC-10301e20
python
# Load data using the refactored data loader (prefers processed CSVs).
from eda.data_loader import load_data

# Optionally pass a path: load_data('data/processed/products_latest.csv')
df = load_data()
print(f'Loaded data: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()
        'Construction Materials': (500, 8000),
        'Auto Parts': (300, 6000)
    }

    prices, min_orders, ratings, review_counts = [], [], [], []
    for cat in chosen_cats:
        lo, hi = base_prices[cat]
        prices.append(round(np.random.uniform(lo, hi), 2))
        min_orders.append(np.random.choice([1, 5, 10, 25, 50, 100]))
        ratings.append(round(np.random.uniform(3.0, 5.0), 1))
        review_counts.append(np.random.randint(0, 500))

    suppliers = [f'Supplier_{i:03d}' for i in np.random.randint(1, 80, n)]

    df = pd.DataFrame({
        'product_name': [f'{cat.split()[0]} Product {i+1}' for i, cat in enumerate(chosen_cats)],
        'category': chosen_cats,
        'price_inr': prices,
        'unit': np.random.choice(units, n),
        'min_order_qty': min_orders,
        'supplier_name': suppliers,
        'supplier_city': np.random.choice(cities, n),
        'rating': ratings,
        'review_count': review_counts,
        'is_verified_supplier': np.random.choice([True, False], n, p=[0.6, 0.4])
    })
    return df

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f'Loaded scraped data: {df.shape[0]} rows, {df.shape[1]} columns')
else:
    df = generate_demo_data(300)
    print('Scraped CSV not found — using demo data (300 records).')
    print(f'Shape: {df.shape}')

df.head()

## 3. Data Overview

In [ ]:
print('=== SHAPE ===')
print(f'{df.shape[0]} rows x {df.shape[1]} columns\n')

print('=== DATA TYPES ===')
print(df.dtypes)

print('\n=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})[missing > 0])
if missing.sum() == 0:
    print('No missing values found.')

In [ ]:
print('=== STATISTICAL SUMMARY ===')
df.describe(include='all').T

## 4. Price Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['price_inr'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (INR)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price (INR)')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

# Log scale for skewed data
axes[1].hist(np.log1p(df['price_inr']), bins=40, color='coral', edgecolor='white')
axes[1].set_title('Price Distribution (Log Scale)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean Price:   ₹{df['price_inr'].mean():,.2f}")
print(f"Median Price: ₹{df['price_inr'].median():,.2f}")
print(f"Price Range:  ₹{df['price_inr'].min():,.2f} — ₹{df['price_inr'].max():,.2f}")

## 5. Category Analysis

In [ ]:
cat_counts = df['category'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Product count per category
cat_counts.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Product Count by Category', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Products')
axes[0].invert_yaxis()

# Avg price per category
cat_price = df.groupby('category')['price_inr'].mean().sort_values(ascending=True)
cat_price.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Average Price by Category (INR)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Avg Price (INR)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

plt.tight_layout()
plt.savefig('category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Supplier & Geographic Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top supplier cities
city_counts = df['supplier_city'].value_counts().head(10)
city_counts.plot(kind='bar', ax=axes[0], color='mediumseagreen', edgecolor='white')
axes[0].set_title('Top Supplier Cities', fontsize=13, fontweight='bold')
axes[0].set_xlabel('City')
axes[0].set_ylabel('Number of Listings')
axes[0].tick_params(axis='x', rotation=30)

# Verified vs non-verified suppliers
if 'is_verified_supplier' in df.columns:
    verified_counts = df['is_verified_supplier'].value_counts()
    axes[1].pie(
        verified_counts,
        labels=['Verified', 'Not Verified'],
        autopct='%1.1f%%',
        colors=['mediumseagreen', 'lightcoral'],
        startangle=90
    )
    axes[1].set_title('Verified vs Unverified Suppliers', fontsize=13, fontweight='bold')
else:
    axes[1].axis('off')

plt.tight_layout()
plt.savefig('supplier_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Price vs Category — Boxplot

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

order = df.groupby('category')['price_inr'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='category', y='price_inr', order=order, ax=ax, palette='muted')

ax.set_title('Price Distribution by Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Price (INR)')
ax.tick_params(axis='x', rotation=30)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

plt.tight_layout()
plt.savefig('price_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Rating Analysis

In [ ]:
if 'rating' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Rating distribution
    axes[0].hist(df['rating'].dropna(), bins=20, color='gold', edgecolor='white')
    axes[0].set_title('Rating Distribution', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Rating')
    axes[0].set_ylabel('Count')

    # Rating vs Price scatter
    axes[1].scatter(df['rating'], df['price_inr'], alpha=0.4, color='steelblue', s=30)
    axes[1].set_title('Rating vs Price', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Rating')
    axes[1].set_ylabel('Price (INR)')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

    plt.tight_layout()
    plt.savefig('rating_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Average Rating: {df['rating'].mean():.2f}")
    print(f"Products rated 4.5+: {(df['rating'] >= 4.5).sum()} ({(df['rating'] >= 4.5).mean()*100:.1f}%)")

## 9. Correlation Heatmap

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, ax=ax,
                linewidths=0.5, cbar_kws={'shrink': 0.8})
    ax.set_title('Correlation Heatmap — Numeric Features', fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Key Findings & Business Insights

In [ ]:
top_cat = df['category'].value_counts().idxmax()
most_expensive_cat = df.groupby('category')['price_inr'].mean().idxmax()
top_city = df['supplier_city'].value_counts().idxmax() if 'supplier_city' in df.columns else 'N/A'
verified_pct = df['is_verified_supplier'].mean() * 100 if 'is_verified_supplier' in df.columns else None
avg_price = df['price_inr'].mean()
price_std = df['price_inr'].std()

print('=' * 55)
print('     KEY FINDINGS — SLOOZE DATA ENGINEER CHALLENGE')
print('=' * 55)
print(f'\n  Total Products Analysed : {len(df)}')
print(f'  Total Unique Categories : {df["category"].nunique()}')
print(f'  Total Unique Suppliers  : {df["supplier_name"].nunique()}')
print(f'\n  Most Listed Category    : {top_cat}')
print(f'  Highest Avg Price Cat   : {most_expensive_cat}')
print(f'  Top Supplier City       : {top_city}')
print(f'\n  Average Product Price   : ₹{avg_price:,.2f}')
print(f'  Price Std Deviation     : ₹{price_std:,.2f}')
if verified_pct:
    print(f'  Verified Suppliers      : {verified_pct:.1f}%')

print('\n  INSIGHTS:')
print(f'  1. Price distribution is heavily right-skewed —')
print(f'     median (₹{df["price_inr"].median():,.0f}) << mean (₹{avg_price:,.0f}),')
print(f'     indicating a few high-value listings inflate the average.')
print(f'  2. {top_cat} dominates listings — high supplier competition.')
print(f'  3. {top_city} is the leading supplier hub by listing volume.')
if verified_pct:
    print(f'  4. {verified_pct:.0f}% verified suppliers — trust signal for buyers.')
print('=' * 55)

## 11. Export Clean Dataset

In [ ]:
df.to_csv('indiamart_clean.csv', index=False)
print(f'Clean dataset exported: indiamart_clean.csv ({len(df)} rows)')